# 🎬 Movie Recommendation System v2

An upgrade of the original content-based recommender. This notebook builds every artifact the new Streamlit app needs.

| # | Upgrade | Where |
|---|---------|-------|
| 1 | Posters, movie details & worldwide search (TMDB API) | Section 9, `app.py`, `tmdb_live.py` |
| 2 | TF-IDF instead of CountVectorizer | Section 3 |
| 3 | Hybrid ranking (similarity + rating/popularity) | Sections 4, 7 |
| 4 | Filters (genre, rating, language, year) | `recommender.py` + `app.py` |
| 5 | Semantic embeddings (sentence-transformers) | Section 3 |
| 6 | Quantitative evaluation (Precision@10, Hit-rate@10) | Section 8 |
| 7 | Merge on movie id instead of title | Section 2 |
| 8 | Collaborative filtering (MovieLens) + hybrid | Sections 5, 7 |

**Before running:** put `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` in a folder called `datasets/` next to this notebook.
MovieLens is downloaded automatically (it needs internet). On Google Colab, run the cells top to bottom.

## 1. Setup

In [ ]:
!pip -q install numpy pandas scipy scikit-learn nltk sentence-transformers matplotlib requests

In [ ]:
import os, re, json, ast, random, zipfile, io, urllib.request
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt
from itertools import product
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_DIR = "datasets"      # folder with the two TMDB csv files
ART_DIR = "artifacts"      # everything the app needs is saved here
os.makedirs(ART_DIR, exist_ok=True)

## 2. Load & merge the TMDB data (upgrade 7)
The old notebook merged the two files on **title**. Some titles appear more than once, which silently created extra rows (4809 instead of 4803).
Here we merge on the unique movie **id** and assert it is a clean one-to-one match.

In [ ]:
movies = pd.read_csv(f"{DATA_DIR}/tmdb_5000_movies.csv")
credits = pd.read_csv(f"{DATA_DIR}/tmdb_5000_credits.csv", engine="python")
print("movies:", movies.shape, "| credits:", credits.shape)

credits = credits.rename(columns={"movie_id": "id"}).drop(columns=["title"])
df = movies.merge(credits, on="id", how="inner", validate="one_to_one")
print("merged on id:", df.shape)          # same number of rows as the movies file

df["overview"] = df["overview"].fillna("")            # keep movies without an overview
df = df.dropna(subset=["release_date"]).reset_index(drop=True)
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype(int)
print("movies kept:", len(df))

## 3. Feature engineering
We build three text representations of every movie:

* **Bag-of-Words** (the original method, kept as a baseline)
* **TF-IDF** (upgrade 2): down-weights words that appear in many movies
* **Sentence embeddings** (upgrade 5): a neural model that understands meaning, so "spaceship" and "starship" are close

In [ ]:
def parse(s):
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return []

df["genres_l"]   = df["genres"].apply(lambda s: [d["name"] for d in parse(s)])
df["keywords_l"] = df["keywords"].apply(lambda s: [d["name"] for d in parse(s)])
df["cast_l"]     = df["cast"].apply(lambda s: [d["name"] for d in parse(s)][:3])          # top 3 actors
df["director"]   = df["crew"].apply(lambda s: next((d["name"] for d in parse(s) if d.get("job") == "Director"), ""))

ps = PorterStemmer()
squash = lambda xs: [x.replace(" ", "") for x in xs]       # "Sam Worthington" -> "SamWorthington"

def make_tags(r):
    toks = r["overview"].split() + squash(r["genres_l"]) + squash(r["keywords_l"]) + squash(r["cast_l"]) + squash([r["director"]] if r["director"] else [])
    return " ".join(ps.stem(t) for t in " ".join(toks).lower().split())

df["tags"] = df.apply(make_tags, axis=1)

def rich_text(r):   # natural-language text for the embedding model (no stemming, keeps spaces)
    return (f"{r['title']}. Genres: {', '.join(r['genres_l'])}. {r['overview']} "
            f"Keywords: {', '.join(r['keywords_l'])}. Directed by {r['director']}. Starring {', '.join(r['cast_l'])}.")

df["text"] = df.apply(rich_text, axis=1)
df[["title", "tags"]].head(3)

In [ ]:
# --- Bag-of-Words (baseline) and TF-IDF (upgrade 2); rows are L2-normalised so dot product = cosine similarity
X_count = normalize(CountVectorizer(max_features=5000, stop_words="english").fit_transform(df["tags"]))
X_tfidf = TfidfVectorizer(max_features=5000, stop_words="english", sublinear_tf=True).fit_transform(df["tags"])
X_count, X_tfidf = sp.csr_matrix(X_count), sp.csr_matrix(normalize(X_tfidf))
print("count:", X_count.shape, "| tfidf:", X_tfidf.shape)

In [ ]:
# --- Sentence embeddings (upgrade 5). Takes ~1-3 minutes on CPU the first time, then it is cached.
emb_path = f"{ART_DIR}/emb_cache.npy"
if os.path.exists(emb_path):
    E = np.load(emb_path)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    E = model.encode(df["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(emb_path, E)
E = np.asarray(E, dtype=np.float32)
print("embeddings:", E.shape)

## 4. Quality prior: rating + popularity (upgrade 3)
A movie with a single 10/10 vote should not beat a movie with thousands of 8/10 votes. We use the IMDb-style **weighted rating**
and combine it with log-popularity into one 0-1 score that the hybrid ranker can use.

In [ ]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.70)
v, R = df["vote_count"], df["vote_average"]
df["weighted_rating"] = (v / (v + m)) * R + (m / (v + m)) * C

mm = lambda s: (s - s.min()) / (s.max() - s.min())
df["quality"] = 0.7 * mm(df["weighted_rating"]) + 0.3 * mm(np.log1p(df["popularity"]))

df["label"] = df["title"] + " (" + df["year"].astype(str) + ")"
dup = df["label"].duplicated(keep=False)
df.loc[dup, "label"] = df.loc[dup, "label"] + " [id " + df.loc[dup, "id"].astype(str) + "]"   # unique names for the dropdown

meta = df[["id", "title", "label", "year", "genres_l", "keywords_l", "cast_l", "director", "overview",
           "vote_average", "vote_count", "popularity", "original_language", "quality"]].copy()
meta.head(3)

## 5. Collaborative filtering with MovieLens (upgrade 8)
TMDB has no user ratings, so we use the free **MovieLens (small)** dataset (about 100,000 ratings from 610 users) and link it to our movies through the TMDB id.
Movies are described by *who liked them*: we factorise the user x movie rating matrix with SVD, then compare movies in that latent space.

In [ ]:
ML_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
if not os.path.exists("ml-latest-small"):
    with urllib.request.urlopen(ML_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")

ratings = pd.read_csv("ml-latest-small/ratings.csv")
links = pd.read_csv("ml-latest-small/links.csv").dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)

id2idx = {mid: i for i, mid in enumerate(df["id"])}
links["idx"] = links["tmdbId"].map(id2idx)
links = links.dropna(subset=["idx"])
ratings = ratings.merge(links[["movieId", "idx"]], on="movieId")
ratings["idx"] = ratings["idx"].astype(int)
print(f"{len(ratings):,} ratings on {ratings['idx'].nunique():,} of our {len(df):,} movies from {ratings['userId'].nunique()} users")

In [ ]:
def build_cf(ratings_df, user_ids, n_items, k=50, min_ratings=5, seed=SEED):
    # Returns L2-normalised item factors (N x k) and a mask of movies that have enough ratings.
    r = ratings_df[ratings_df["userId"].isin(user_ids)].copy()
    r["c"] = r["rating"] - r.groupby("userId")["rating"].transform("mean")        # remove each user's rating bias
    umap = {u: i for i, u in enumerate(sorted(set(user_ids)))}
    M = sp.csr_matrix((r["c"].values, (r["userId"].map(umap).values, r["idx"].values)), shape=(len(umap), n_items))
    has = np.bincount(r["idx"].values, minlength=n_items) >= min_ratings
    k = max(2, min(k, M.shape[0] - 1, M.shape[1] - 1))
    svd = TruncatedSVD(n_components=k, random_state=seed).fit(M)
    F = normalize(svd.components_.T * np.sqrt(svd.singular_values_))
    F[~has] = 0
    return F.astype(np.float32), has

# --- split USERS so the evaluation is fair: 60% build the CF model, 20% tune the hybrid, 20% test
users = np.array(sorted(ratings["userId"].unique()))
np.random.RandomState(SEED).shuffle(users)
a, b = int(0.6 * len(users)), int(0.8 * len(users))
train_u, val_u, test_u = users[:a], users[a:b], users[b:]
cf_train, has_train = build_cf(ratings, train_u, len(df))
print("movies with enough ratings for CF:", int(has_train.sum()))

## 6. The recommender class (upgrades 4 and 8)
The runtime code lives in `recommender.py` so the notebook and the Streamlit app use **exactly the same** logic. It supports:
`quality` (popularity baseline), `count`, `tfidf`, `embed`, `cf` and `hybrid`, plus filters by genre, minimum rating, language and release year.
Similarities are computed on the fly for one movie, so no giant matrix needs to be stored.

In [ ]:
%%writefile recommender.py
"""Runtime recommender shared by the notebook (evaluation) and the Streamlit app.

It only needs numpy, pandas and scipy, so the deployed app stays light.
Everything is computed on the fly for ONE query movie, so no N x N similarity
matrix has to be stored (the old project pickled a ~90 MB matrix).
"""
import json
import os

import numpy as np
import pandas as pd
import scipy.sparse as sp

METHODS = ["quality", "count", "tfidf", "embed", "cf", "hybrid"]
DEFAULT_WEIGHTS = {"tfidf": 0.25, "embed": 0.35, "cf": 0.30, "quality": 0.10}
LIST_COLS = ["genres_l", "keywords_l", "cast_l"]


def _minmax(v, mask):
    """Scale scores to 0-1 using only the candidate movies (mask == True)."""
    if not mask.any():
        return np.zeros_like(v, dtype=float)
    x = np.where(mask, v, np.nan)
    lo, hi = np.nanmin(x), np.nanmax(x)
    out = (v - lo) / (hi - lo) if hi > lo else np.zeros_like(v, dtype=float)
    return np.where(mask, out, 0.0)


def top_n(scores, n):
    """Indices of the n best scores (descending); -inf entries are dropped."""
    n = max(1, min(n, len(scores) - 1))
    idx = np.argpartition(-scores, n - 1)[:n]
    idx = idx[np.argsort(-scores[idx])]
    return idx[np.isfinite(scores[idx])]


class Recommender:
    def __init__(self, meta, count, tfidf, emb, cf, has_cf, weights=None):
        self.meta = meta.reset_index(drop=True)
        self.count = sp.csr_matrix(count)      # L2-normalised Bag-of-Words
        self.tfidf = sp.csr_matrix(tfidf)      # L2-normalised TF-IDF
        self.emb = np.asarray(emb, dtype=np.float32)   # L2-normalised sentence embeddings
        self.cf = np.asarray(cf, dtype=np.float32)     # L2-normalised item factors (0 rows = no ratings)
        self.has_cf = np.asarray(has_cf, dtype=bool)
        self.weights = dict(weights or DEFAULT_WEIGHTS)
        self.quality = self.meta["quality"].to_numpy(float)
        self.labels = self.meta["label"].tolist()
        self.label_to_idx = {l: i for i, l in enumerate(self.labels)}
        self.n = len(self.meta)

    # ---------- save / load ----------
    def save(self, art_dir="artifacts"):
        os.makedirs(art_dir, exist_ok=True)
        m = self.meta.copy()
        for c in LIST_COLS:                      # lists -> "a|b|c" (robust across pandas versions)
            m[c] = m[c].apply(lambda xs: "|".join(xs))
        m.to_csv(f"{art_dir}/meta.csv.gz", index=False)
        sp.save_npz(f"{art_dir}/count.npz", self.count)
        sp.save_npz(f"{art_dir}/tfidf.npz", self.tfidf)
        np.save(f"{art_dir}/emb.npy", self.emb)
        np.save(f"{art_dir}/cf.npy", self.cf)
        np.save(f"{art_dir}/has_cf.npy", self.has_cf)
        with open(f"{art_dir}/weights.json", "w") as f:
            json.dump(self.weights, f, indent=2)

    @classmethod
    def load(cls, art_dir="artifacts"):
        meta = pd.read_csv(f"{art_dir}/meta.csv.gz")
        for c in LIST_COLS:
            meta[c] = meta[c].fillna("").apply(lambda s: [x for x in s.split("|") if x])
        meta["overview"] = meta["overview"].fillna("")
        meta["director"] = meta["director"].fillna("")
        weights = None
        if os.path.exists(f"{art_dir}/weights.json"):
            with open(f"{art_dir}/weights.json") as f:
                weights = json.load(f)
        return cls(meta, sp.load_npz(f"{art_dir}/count.npz"), sp.load_npz(f"{art_dir}/tfidf.npz"),
                   np.load(f"{art_dir}/emb.npy"), np.load(f"{art_dir}/cf.npy"),
                   np.load(f"{art_dir}/has_cf.npy"), weights)

    # ---------- scoring ----------
    def components(self, i):
        """Raw similarity of every movie to movie i, for each signal."""
        return {
            "count": (self.count @ self.count[i].T).toarray().ravel(),
            "tfidf": (self.tfidf @ self.tfidf[i].T).toarray().ravel(),
            "embed": self.emb @ self.emb[i],
            "cf": self.cf @ self.cf[i],
        }

    def filter_mask(self, genres=None, min_rating=0.0, language=None, year_range=None, pool=None):
        m = np.ones(self.n, dtype=bool) if pool is None else pool.copy()
        if genres:
            wanted = set(genres)
            m &= self.meta["genres_l"].apply(lambda g: bool(wanted.intersection(g))).to_numpy()
        if min_rating:
            m &= (self.meta["vote_average"] >= min_rating).to_numpy()
        if language and language != "Any":
            m &= (self.meta["original_language"] == language).to_numpy()
        if year_range:
            m &= self.meta["year"].between(year_range[0], year_range[1]).to_numpy()
        return m

    def score(self, i, method="hybrid", weights=None, mask=None, comps=None):
        mask = np.ones(self.n, dtype=bool) if mask is None else mask.copy()
        mask[i] = False                                  # never recommend the query itself
        if method == "quality":
            return np.where(mask, self.quality, -np.inf)
        comps = comps or self.components(i)
        if method != "hybrid":
            return np.where(mask, comps[method], -np.inf)
        w = dict(weights or self.weights)
        if not self.has_cf[i]:                           # query has no rating data -> drop the CF signal
            w["cf"] = 0.0
        total = sum(w.values())
        if total <= 0:
            return np.where(mask, self.quality, -np.inf)
        s = np.zeros(self.n)
        for k in ("tfidf", "embed", "cf"):
            if w.get(k, 0) > 0:
                v = np.clip(comps[k], 0, None) if k == "cf" else comps[k]
                s += (w[k] / total) * _minmax(v, mask)
        s += (w.get("quality", 0) / total) * self.quality
        return np.where(mask, s, -np.inf)

    def recommend(self, movie, n=10, method="hybrid", weights=None, **filters):
        """movie = label string or integer index. filters: genres, min_rating, language, year_range."""
        i = self.label_to_idx[movie] if isinstance(movie, str) else int(movie)
        comps = self.components(i)
        mask = self.filter_mask(**filters)
        s = self.score(i, method, weights, mask, comps)
        top = top_n(s, n)
        out = self.meta.iloc[top].copy()
        out["score"] = s[top]
        for k, v in comps.items():
            out[f"s_{k}"] = v[top]
        return out

In [ ]:
from recommender import Recommender, top_n, METHODS, DEFAULT_WEIGHTS, _minmax

rec_eval = Recommender(meta, X_count, X_tfidf, E, cf_train, has_train)
print("ready:", rec_eval.n, "movies")

## 7. Offline evaluation and hybrid tuning (upgrade 6)
**Protocol.** For each held-out user we take movies they *liked* (rating >= 4). We pick a liked movie as the **query**, ask the system for 10 recommendations and check how many of the user's *other* liked movies appear.

* **Precision@10**: share of the 10 recommendations that the user liked
* **Hit-rate@10**: share of queries where at least one recommendation was liked
* **Genre Jaccard@10**: genre overlap between query and recommendations (a content-only sanity check; it favours methods that use genres, so treat it as secondary)

Only movies with enough ratings are candidates, so every method competes on the same pool. Absolute numbers look small because each user liked only a few of the thousands of candidates; compare methods **relative to each other**.
The hybrid weights are tuned on the **validation** users and everything is reported on the unseen **test** users.

In [ ]:
pool = has_train.copy()                                   # candidate movies (enough ratings)
gsets = [set(g) for g in meta["genres_l"]]
jacc = lambda a, b: len(a & b) / len(a | b) if (a | b) else 0.0

def build_queries(user_ids, per_user=3, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    sub = ratings[ratings["userId"].isin(user_ids)]
    for u, g in sub.groupby("userId"):
        liked = [i for i in g.loc[g["rating"] >= 4, "idx"].unique() if pool[i]]
        if len(liked) < 6:
            continue
        for q in rng.choice(liked, size=min(per_user, len(liked)), replace=False):
            out.append((int(q), set(int(x) for x in liked) - {int(q)}))
    return out

val_q, test_q = build_queries(val_u), build_queries(test_u)
val_c  = [rec_eval.components(q) for q, _ in val_q]
test_c = [rec_eval.components(q) for q, _ in test_q]
print("validation queries:", len(val_q), "| test queries:", len(test_q))

def evaluate(queries, comps_list, method, weights=None, K=10, genre=True):
    P = H = J = 0.0
    for (q, rel), comps in zip(queries, comps_list):
        top = top_n(rec_eval.score(q, method, weights, pool, comps), K)
        hits = len(rel.intersection(top.tolist()))
        P += hits / K; H += hits > 0
        if genre:
            J += np.mean([jacc(gsets[q], gsets[t]) for t in top])
    n = len(queries)
    return {"Precision@10": P / n, "HitRate@10": H / n, "GenreJaccard@10": J / n if genre else np.nan}

In [ ]:
# --- tune the hybrid weights on the VALIDATION users (grid search, weights sum to 1)
def simplex_grid(keys, step=0.1):
    k = int(round(1 / step))
    for combo in product(range(k + 1), repeat=len(keys) - 1):
        if sum(combo) <= k:
            vals = list(combo) + [k - sum(combo)]
            yield {key: v / k for key, v in zip(keys, vals)}

best_w, best = None, (-1, -1)
for w in simplex_grid(["tfidf", "embed", "cf", "quality"]):
    r = evaluate(val_q, val_c, "hybrid", w, genre=False)
    if (r["Precision@10"], r["HitRate@10"]) > best:
        best, best_w = (r["Precision@10"], r["HitRate@10"]), w
print("best hybrid weights:", best_w, "| validation Precision@10 = %.4f" % best[0])

In [ ]:
# --- final comparison on the unseen TEST users
names = {"quality": "Popularity only (baseline)", "count": "Bag-of-Words (original project)", "tfidf": "TF-IDF",
         "embed": "Sentence embeddings", "cf": "Collaborative filtering (SVD)", "hybrid": "Hybrid (tuned)"}
rows = []
for m_ in ["quality", "count", "tfidf", "embed", "cf", "hybrid"]:
    r = evaluate(test_q, test_c, m_, best_w if m_ == "hybrid" else None)
    rows.append({"Method": names[m_], **r})
results = pd.DataFrame(rows).set_index("Method")
results.to_csv("evaluation_results.csv")
results.round(4)

In [ ]:
ax = results[["Precision@10", "HitRate@10"]].plot(kind="bar", figsize=(9, 4.5), rot=25, color=["#4F81BD", "#C0504D"])
ax.set_title("Recommendation quality on unseen users (higher is better)")
ax.set_ylabel("score"); plt.tight_layout(); plt.savefig("evaluation_chart.png", dpi=200); plt.show()

## 8. Build the final model and save the artifacts
Now that the design is validated, the collaborative model is **refit on all users** (more data) and everything the app needs is saved in `artifacts/`.

In [ ]:
cf_all, has_all = build_cf(ratings, users, len(df))
final = Recommender(meta, X_count, X_tfidf, E, cf_all, has_all, weights=best_w)
final.save(ART_DIR)
if os.path.exists(f"{ART_DIR}/emb_cache.npy"):
    os.remove(f"{ART_DIR}/emb_cache.npy")      # the app uses emb.npy, so the cache copy is not needed
for f in sorted(os.listdir(ART_DIR)):
    print(f"{f:16s} {os.path.getsize(os.path.join(ART_DIR, f)) / 1e6:6.2f} MB")

In [ ]:
# --- try it: same movie, every engine
title = "The Dark Knight Rises"
q = meta.index[meta["title"] == title][0]
for m_ in ["count", "tfidf", "embed", "cf", "hybrid"]:
    out = final.recommend(int(q), n=5, method=m_)
    print(f"{names[m_]:35s} ->", ", ".join(out["title"]))

print("\nHybrid with filters (Action/Thriller, rating >= 7, 2005 or later):")
final.recommend(int(q), n=5, genres=["Action", "Thriller"], min_rating=7, year_range=(2005, 2030))[["title", "year", "vote_average", "score"]]

## 9. Posters and details from TMDB (upgrade 1)
The app fetches posters at runtime with the TMDB API. **Never put the API key in your code or GitHub.**
Create a new key at themoviedb.org (Settings > API), then:

* **Local / Colab:** set an environment variable `TMDB_API_KEY`
* **Streamlit Cloud:** App settings > Secrets, add `TMDB_API_KEY = "your_key"`

> The key that was in your old `app.py` (commented out) is public on GitHub. Treat it as leaked and regenerate it.

In [ ]:
import requests
from IPython.display import Image, display

def poster_url(movie_id, key):
    r = requests.get(f"https://api.themoviedb.org/3/movie/{int(movie_id)}", params={"api_key": key}, timeout=8)
    r.raise_for_status()
    p = r.json().get("poster_path")
    return f"https://image.tmdb.org/t/p/w342{p}" if p else None

key = os.environ.get("TMDB_API_KEY")
if key:
    for mid in final.recommend(int(q), n=3)["id"]:
        display(Image(url=poster_url(mid, key)))
else:
    print("Set TMDB_API_KEY to preview posters. The Streamlit app does this automatically.")

## 10. Run the app
Copy these into your GitHub repository (replace the old `app.py`, `requirements.txt`, and delete the old `.pkl` files):

```
app.py
recommender.py
tmdb_live.py        <- worldwide search + live recommendations (any movie on TMDB)
requirements.txt
artifacts/          <- created by this notebook
```
Then run `streamlit run app.py` (or redeploy on Streamlit Cloud). Add `TMDB_API_KEY` to the app secrets to see posters and unlock worldwide search.

**Two modes in the app**

* **All movies in the world (live TMDB):** search any title, and recommendations come from TMDB's related-movie data, re-ranked by genre overlap, quality and language (see `tmdb_live.py`). Needs the API key.
* **Curated catalog:** the 4,800 movies covered by the models trained in this notebook (hybrid, embeddings, TF-IDF, collaborative).